#### CS/ECE/ISyE 524 — Introduction to Optimization — Spring 2025

# Blood Transfusion Supply and Demand

#### By: Shawn Lin


### 1. Introduction
At the beginning of this semester, the American Red Cross partnered with the University of Wisconsin–Madison for Bucky’s Blood Drive[1]. Usually, when a blood donation event is organized, it often indicates that the local blood bank is running low on supply. A blood transfusion is a medical procedure where blood or its components are given to a patient through a vein to replace lost or deficient blood. These transfusions are crucial in all kinds of cases such as trauma, surgeries, and cancer treatment. We often see in TV medical shows lines like, “The patient has blood type A, but we don’t have type A or O available.” While that is scripted drama, it actually reflects a real problem hospitals face with blood type compatibility and supply issues. In practice, blood transfusions follow strict compatibility rules. The blood transfusions chart (shown in figure 1) outlines which blood types can be safely donated to or received by others[2]. To make my model more realistic, each general blood types (A, B, O, AB) will be further split into positive and negative (i.e. A+, A-, B+. B-, …etc). It is not easy for us to determine if the supply met the demand when the numbers are large. This leads to the central question of the project: **How can we efficiently determine whether a hospital’s current blood supply is sufficient to meet patient demand, given these compatibility constraints?** A couple assumptions I made: 1. Supply and demand are measured in units of days and are whole numbers. 2. Transfusions are only considered from donor to recipient based on compatibility not patient prioritization.

<div style="text-align: center;">
  <div style="display: inline-block;">
    <img src="chart.png" alt="Blood Type Compatibility Chart" width="400"/>
    <p style="text-align: center;"><strong>Figure 1:</strong> Blood Type Compatibility Chart</p>
  </div>
</div>

### 2. Mathematical Model
This problem can be modeled as a maximum flow problem on a bipartite graph, where we seek to determine whether the available blood supply can meet the demand under blood-type compatibility constraints. If the maximum flow is equal to the total demand, then we can conclude that current blood supply is sufficient to meet the demand.

#### 2.1 Sets and Parameters

Let:

$$
B = \text{set of all blood types} = \{A^-, A^+, B^-, B^+, O^-, O^+, AB^-, AB^+\}
$$

$$
s_b \text{ be the supply of blood type } b \in B.
$$

$$
d_b \text{ be the demand for blood type } b \in B.
$$

$$
C = \{ (i, j) \in B \times B \mid \text{blood type } i \text{ can be safely transfused to recipient of type } j \}
$$


#### 2.2 Decision Variable
Let: 

$$
x_{i,j} \text{ be the number of units of blood transferred from supply type i to demand type j}
$$

#### 2.3 Objective Function

$$
\text{Maximize} \quad \sum_{i \in B} \sum_{j \in B} x_{i,j}
$$

#### 2.4 Constraints

**Supply constraint:** The amount sent out cannot exceed the available supply.

$$
\sum_{j \in B} x_{i,j} \leq s_i \quad \forall i \in B
$$

**Demand constraint:** The amount received must not exceed the demand of each blood type.

$$
\sum_{i \in B} x_{i,j} \leq d_j \quad \forall j \in B
$$

**Compatibility constraint:** Incompatible blood transfusions are not allowed

$$
x_{i,j} = 0 \quad \text{if } (i, j) \notin C
$$

**Non-negativity and discreteness constraint:** The number of units transferred must be a positive integer.

$$
x_{i,j} \in \mathbb{Z}_{\geq 0} \quad \forall (i, j) \in B \times B
$$


### 3. Solution

For security reasons, most hospitals in the United States do not disclose their blood bank inventory levels. However, they may provide a rough estimate of how many days the blood bank can operate without receiving new donations. The data provided below is from Canadian Blood Services, which maintains a "days of inventory on hand" metric for each blood group, updated regularly on their website [3]. For simplicity, we assume that one day of inventory corresponds to one unit of supply.

Given supply:

<div style="text-align: center;">
  <div style="display: inline-block;">
    <img src="inventory.png" alt="Inventory"/>
    <p style="text-align: center;"><strong>Figure 2:</strong> Real-time Blood Type Inventory</p>
  </div>
</div>

On the demand side, there is no reliable estimate of the current demand for each blood type. However, blood system inventory management best practice guidelines suggest that a well-functioning hospital should maintain at least 4 to 6 days of inventory [4]. So, we could set the demand to 4–6 units (values to be generated) for each blood type.

In [3]:
using Random

# Set the random seed for reproducibility
Random.seed!(12)

# Generate 8 random integers between 4 and 6
demand = rand(4:6, 8)

# This is our "fake" demand
println(demand)
println("Total demand: ", sum(demand))

[4, 6, 5, 6, 5, 4, 4, 6]
Total demand: 40


In [4]:
using JuMP, HiGHS

# Sets and Parameters
B = ["A+" "A-" "B+" "B-" "O+" "O-" "AB+" "AB-"]
supply = Dict("A+" => 9, "A-" => 7, "B+" => 12, "B-" => 4, 
              "O+" => 4, "O-" => 8, "AB+" => 24, "AB-" => 15)
demand = Dict("A+" => 4, "A-" => 6, "B+" => 5, "B-" => 6, 
              "O+" => 5, "O-" => 4, "AB+" => 4, "AB-" => 6)

# Compatibility map (i.e., donor => list of compatible recipients)
compatible = Dict(
    "A+"  => ["A+", "AB+"],
    "A-"  => ["A+", "A-", "AB+", "AB-"],
    "B+"  => ["B+", "AB+"],
    "B-"  => ["B+", "B-", "AB+", "AB-"],
    "O+"  => ["A+", "B+", "O+", "AB+"],
    "O-"  => ["A+", "A-", "B+", "B-", "O+", "O-", "AB+", "AB-"],
    "AB+" => ["AB+"],
    "AB-" => ["AB+", "AB-"]
)

# Model
m = Model(HiGHS.Optimizer)
set_silent(m)

# Decision Variables
@variable(m, x[i in B, j in B] >= 0, Int)

# Objective Function: Maximize total units transfused
@objective(m, Max, sum(x[i, j] for i in B, j in B))

# Supply constraints: 
# The amount sent out cannot exceed the available supply.
@constraint(m, [i in B], sum(x[i, j] for j in B) <= supply[i])

# Demand constraints: 
# The amount received must not exceed the demand of each blood type.
@constraint(m, [j in B], sum(x[i, j] for i in B) <= demand[j])

# Compatibility constraints: 
# Cannot transfused if blood type i is not compatible with j
for i in B, j in B
    if j ∉ compatible[i]
        @constraint(m, x[i, j] == 0)
    end
end

optimize!(m)

# Print solution
println("Objective value (total transfused, max flow): ", objective_value(m))
# Iterate through supply and demand pairs. 
# If there is any transfusion, print the result
for i in B, j in B
    val = value(x[i, j])
    if val != 0
        print("Transfer ", val, " units from ", i, "(supply) to ")
        println(j, "(demand)")
    end
end


Objective value (total transfused, max flow): 40.0
Transfer 4.0 units from A+(supply) to A+(demand)
Transfer 4.0 units from A+(supply) to AB+(demand)
Transfer 6.0 units from A-(supply) to A-(demand)
Transfer 5.0 units from B+(supply) to B+(demand)
Transfer 4.0 units from B-(supply) to B-(demand)
Transfer 3.0 units from O+(supply) to O+(demand)
Transfer 2.0 units from O-(supply) to B-(demand)
Transfer 2.0 units from O-(supply) to O+(demand)
Transfer 4.0 units from O-(supply) to O-(demand)
Transfer 6.0 units from AB-(supply) to AB-(demand)


### 4. Results, Discussion, Analysis

#### 4.1 Results
The model outputs a maximum flow of 40, which matches the total demand of 40 units across all blood types. That means, given the current inventory levels and compatibility rules, the hospital’s blood inventory can be considered healthy and capable of meeting all demand needs.


#### 4.2 Discussion
From the console output, we can see that the model correctly determines how to reallocate blood supplies from one type to fulfill the demand of compatible types. Although the supply for both B- and O+ is insufficient to meet their respective demands directly, the model utilizes O- supply to cover these gaps in accordance with blood compatibility rules. This demonstrates the model’s effectiveness in leveraging donor-recipient flexibility when it exists.

#### 4.3 Analysis
Even though the current model successfully captures the core dynamics of blood supply allocation based on compatibility, there are a few limitations that needs to discuss. First of all, blood type compatibility is treated as binary—either transfusable or not—while in reality, factors such as cross-matching compatibility, Rh antibodies, and patient-specific conditions may influence transfusion decisions[5]. Secondly, when there is a demand of 4 units for AB+ blood, the model may prioritize selecting supply from A+ rather than AB+ due to the way the optimization objective is constructed. However, in real hospital practice, it is preferable to allocate AB+ supply to AB+ patients first, since AB+ donors can only serve AB+ recipients, whereas A+ donors are also compatible with both A+ and AB+ recipients. 

#### 4.4 Generalizations and Variations
This model can be extended in several meaningful ways to better reflect real-world scenarios: 1. Instead of modeling supply and demand at a single point in time, we can simulate how they change over several days. 2. Instead of focusing on one hospital, we can extend the model to include a network of hospitals sharing blood, with transportation constraints or costs. 3. We can create a model that minimizes the maximum blood shortage across different demand scenarios for various blood types. 4. We can incorporate costs, such as the cost of storing or transporting blood to meet demand.

### 5. Conclusion
In this project, I build a maximum flow model to check if a hospital's current blood inventory is sufficient to meet the demand, a pseudo random generated demand based on best practice guideline, while following blood type compatibility rules. The model worked– it matched supply to demand with a total flow of 40, which was exactly what the demand was in the scenario. The result also confirms that the model is effective in reallocating supply across compatible blood types to meet demand efficiently. 

One possible future direction would be to extend the model into a network with multiple hospitals, where blood can be transported between them. In such a system, transportation costs would need to be considered, and each hospital would have its own supply and demand constraints. This extension would make the model more realistic and applicable to regional or national blood management strategies. If real hospitals were willing to share the true data on supply and demand, that would help a lot with testing and refining the model. It could even be used to help with emergency planning.

### 6. Reference
[1] WKOW Staff. “Bucky’s Blood Drive Set for Jan. 29–30.” WKOW, 23 Feb. 2024, www.wkow.com/news/buckys-blood-drive-set-for-jan-29-30/article_2c08b424-d99b-11ef-b109-9f3d7f813d81.html.

[2] Stanford Blood Center. “Blood Type Compatibility Chart.” Stanford Blood Center, https://stanfordbloodcenter.org/donate-blood/blood-donation-facts/blood-types/0318-southbay-center-infographics_compatibility-web/.

[3] Canadian Blood Services. "Blood for Life." Canadian Blood Services, 2025, https://www.blood.ca/en/blood.

[4] Canadian Blood Services. Inventory Management: Best Practices for Canadian Blood Operators, 2020, www.blood.ca.

[5] "Blood Transfusions and the Immune System." NCBI Bookshelf, U.S. National Library of Medicine, https://www.ncbi.nlm.nih.gov/books/NBK2265/.